# 011 · The Laplace Transform

| | |
|---|---|
| **Track** | Python for Calculus |
| **Level** | 4 — Transforms and differential equations |
| **Prerequisites** | 006 and 009 |
| **Next** | 012 · First-Order Linear Differential Equations |

**Place in the path.** The Laplace transform turns a function of \(t\) into a function of \(s\) by an improper integral.
Differentiation in \(t\) becomes multiplication by \(s\) in the transform world. That is why lesson 012's differential equations
can be solved by algebra once you are fluent here. This lesson builds the fluency and checks it against the integral definition.

## Learning objectives

By the end of this lesson you can:

- write the definition \(\mathcal{L}\{f\}(s) = \int_0^{\infty} e^{-st} f(t)\, dt\)
- compute that integral for \(1\) and for \(t\), then match SymPy's `laplace_transform`
- recall and verify the transforms of \(t^n\), \(e^{at}\), \(\sin(at)\), and \(\cos(at)\)
- use linearity
- invert \(\mathcal{L}\{\sin(at)\}\) and recover the sine


## 1. The definition is an improper integral

**See.** Fix \(s > 0\). Multiply \(f(t)\) by a decaying exponential and integrate from 0 to \(\infty\).

\[
\mathcal{L}\{f\}(s) = \int_0^{\infty} e^{-st} f(t)\, dt
\]

For \(f(t) = 1\):

\[
\int_0^{\infty} e^{-st}\, dt = \left[-\frac{e^{-st}}{s}\right]_0^{\infty} = \frac{1}{s}
\]

The upper limit uses lesson 006: \(e^{-st} \to 0\) as \(t \to \infty\) when \(s > 0\).

**Name.** `sp.laplace_transform(f, t, s)` returns three things: the transform, the half-plane of convergence, and conditions.
This lesson keeps the transform, which is the first entry, and checks it.


In [ ]:
import sympy as sp

t = sp.symbols("t", nonnegative=True)
s = sp.symbols("s", positive=True)
a = sp.symbols("a", positive=True)

from_definition = sp.integrate(sp.exp(-s * t) * 1, (t, 0, sp.oo))
from_library = sp.laplace_transform(1, t, s)[0]
print("integral definition:", from_definition)
print("laplace_transform:", from_library)
assert sp.simplify(from_definition - 1 / s) == 0
assert sp.simplify(from_library - 1 / s) == 0
print("definition and library agree on L{1}")


**Do the same for** \(f(t) = t\). Integration by parts, which you used in lesson 008, gives \(1/s^2\).
Let SymPy do the parts, then compare.


In [ ]:
import sympy as sp

t = sp.symbols("t", nonnegative=True)
s = sp.symbols("s", positive=True)

from_definition = sp.integrate(t * sp.exp(-s * t), (t, 0, sp.oo))
from_library = sp.laplace_transform(t, t, s)[0]
print(from_definition)
assert sp.simplify(from_definition - 1 / s ** 2) == 0
assert sp.simplify(from_library - from_definition) == 0
print("definition and library agree on L{t}")


## 2. A short table you should be able to recompute

**Name.** For \(n\) a positive integer and \(a > 0\), with \(s\) large enough:

\[
\mathcal{L}\{t^n\} = \frac{n!}{s^{n+1}}, \qquad
\mathcal{L}\{e^{at}\} = \frac{1}{s-a}, \qquad
\mathcal{L}\{\sin(at)\} = \frac{a}{s^2+a^2}, \qquad
\mathcal{L}\{\cos(at)\} = \frac{s}{s^2+a^2}
\]

**Do.** Verify \(n = 1, \ldots, 5\) for the power rule, then verify the exponential, sine, and cosine against these formulas.
The library is allowed to return an equivalent expression. `simplify(found - expected) == 0` is the test, not string equality.


In [ ]:
import sympy as sp

t = sp.symbols("t", nonnegative=True)
s = sp.symbols("s", positive=True)
a = sp.symbols("a", positive=True)
n = sp.symbols("n", integer=True, positive=True)

def transform(expression):
    return sp.simplify(sp.laplace_transform(expression, t, s)[0])

for k in range(1, 6):
    found = transform(t ** k)
    expected = sp.factorial(k) / s ** (k + 1)
    print(f"L{{t^{k}}} =", found)
    assert sp.simplify(found - expected) == 0

pairs = [
    (sp.exp(a * t), 1 / (s - a)),
    (sp.sin(a * t), a / (s ** 2 + a ** 2)),
    (sp.cos(a * t), s / (s ** 2 + a ** 2)),
]
for expression, expected in pairs:
    found = transform(expression)
    print(expression, "->", found)
    assert sp.simplify(found - expected) == 0

print("table checks passed")


## 3. Linearity

**See.** The integral of a sum is the sum of the integrals, and constants factor out.
So the transform does too:

\[
\mathcal{L}\{3 + 2t\} = 3\mathcal{L}\{1\} + 2\mathcal{L}\{t\} = \frac{3}{s} + \frac{2}{s^2}
\]

**Check a second way.** Call `laplace_transform` on \(3 + 2t\) directly, and also add the two transforms you already know.


In [ ]:
import sympy as sp

t = sp.symbols("t", nonnegative=True)
s = sp.symbols("s", positive=True)

direct = sp.laplace_transform(3 + 2 * t, t, s)[0]
by_linearity = 3 * sp.laplace_transform(1, t, s)[0] + 2 * sp.laplace_transform(t, t, s)[0]
print(sp.simplify(direct))
assert sp.simplify(direct - by_linearity) == 0
assert sp.simplify(direct - (3 / s + 2 / s ** 2)) == 0
print("linearity agrees with the direct transform")


## 4. One inverse, as a bridge to differential equations

**See.** Solving a differential equation by Laplace transforms ends with an inverse step:
you know \(\mathcal{L}\{y\}\), and you want \(y\).

**Do.** You already computed \(\mathcal{L}\{\sin(at)\} = a/(s^2+a^2)\).
Invert the right-hand side and demand the sine back.

SymPy multiplies by the unit step \(\mathrm{Heaviside}(t)\), because the transform only saw \(t \ge 0\).
For \(t > 0\) that factor is 1. The identity to check is

\[
\mathcal{L}^{-1}\left\{\frac{a}{s^2+a^2}\right\} = \sin(at)\, H(t)
\]


In [ ]:
import sympy as sp

t = sp.symbols("t", real=True)
s = sp.symbols("s", positive=True)
a = sp.symbols("a", positive=True)

inverted = sp.inverse_laplace_transform(a / (s ** 2 + a ** 2), s, t)
expected = sp.sin(a * t) * sp.Heaviside(t)
print(inverted)
assert sp.simplify(inverted - expected) == 0
print("inverse of a/(s**2+a**2) is sin(at) for t > 0")


### Own it

\(\mathcal{L}\{e^{2t}\}\) should be \(1/(s-2)\). This is the exponential rule at \(a = 2\), which is the growth factor in the first differential equation of lesson 012.


In [ ]:
import sympy as sp

t = sp.symbols("t", nonnegative=True)
s = sp.symbols("s", positive=True)
found = sp.laplace_transform(sp.exp(2 * t), t, s)[0]
print(found)
assert sp.simplify(found - 1 / (s - 2)) == 0
print("checkpoint passed")


## Mistakes this lesson is meant to prevent

- Forgetting the third and second return values of `laplace_transform` and printing a tuple as if it were \(F(s)\).
- Using the formulas for \(e^{at}\) and \(\sin(at)\) without the condition that \(s\) is large enough. SymPy's convergence condition is the second return value; read it when the parameters are not obviously positive.
- Dropping \(\mathrm{Heaviside}(t)\) and then evaluating the inverse at a negative time. The transform did not know about \(t < 0\).
- Treating the transform as a derivative. It is an integral. The derivative property comes from integrating by parts, which is a theorem, not the definition.

## What you now have, and what comes next

You can build the basic table from the definition and from the library, and you can invert one entry.
Lesson 012 solves \(y' + P(x)y = Q(x)\) directly with SymPy and checks the solution numerically.
The Laplace table is the reason those closed forms are not magic: a derivative became multiplication, and the inverse brought you home.
